# Plotnine Exercises

This notebook contains several exercises for creating visualizations with Plotnine.
Each exercise includes hints and an answer for when you're stuck.

## Resources

* [Plotnine documentation](https://plotnine.org)
* [Cheat Sheet (PDF)](https://posit.co/wp-content/uploads/2022/10/data-visualization-1.pdf)

In [ ]:
import polars as pl
from plotnine import *

## Read data

You'll be working with a dataset of open-source project metrics.
It contains daily activity counts (downloads, issues, PRs, forks) and totals (views, likes) from sources like GitHub, PyPI, and CRAN.

To keep things manageable, you'll filter to PyPI download counts for a few Python packages.

In [ ]:
metrics = pl.read_parquet("data/metrics.parquet")
metrics

In [ ]:
downloads = metrics.filter(
    pl.col("source") == "pypi",
    pl.col("metric") == "daily_downloads",
)
downloads

## Exercise 1: Mapping columns to aesthetics

Create a line plot showing daily PyPI downloads over time, with `date` on the x-axis and `value` on the y-axis.

**Optional bonus questions:**

* Can you color the lines by `project`?
* What happens if you use `geom_point()` instead of `geom_line()`?

**Hints** *(highlight to reveal)*

<div style="color: #555; background-color: #555; padding: 8px; border-radius: 4px;">
&bull; Use the <code>ggplot()</code> function to start a new Plotnine visualization.<br>
&bull; Use the <code>aes()</code> function to map columns to aesthetics.<br>
&bull; Find the appropriate geometry type by typing <code>geom_&lt;TAB&gt;</code> or check the cheat sheet.<br>
</div>

**Answer** *(highlight to reveal)*

<div style="color: #555; background-color: #555; padding: 8px; border-radius: 4px;">
<pre style="color: inherit; background: inherit; margin: 4px 0; padding: 0;">ggplot(downloads, aes(x="date", y="value", color="project")) + geom_line()</pre>
</div>

## Exercise 2: Statistical transformations

The following code snippet yields an error:

In [ ]:
# ggplot(downloads, aes(x="value", y="project")) + geom_histogram()

What needs to be changed in order to produce a histogram of download counts?

**Hints** *(highlight to reveal)*

<div style="color: #555; background-color: #555; padding: 8px; border-radius: 4px;">
&bull; The function <code>geom_histogram()</code> does a statistical transformation. Under the hood, it calculates the <code>y</code> aesthetic for us.<br>
</div>

**Answer** *(highlight to reveal)*

<div style="color: #555; background-color: #555; padding: 8px; border-radius: 4px;">
<pre style="color: inherit; background: inherit; margin: 4px 0; padding: 0;">ggplot(downloads, aes(x="value")) + geom_histogram()</pre>
</div>

## Exercise 3: From data values to aesthetic values using scales

The stacked bar chart produced by the following snippet has hideous colors. Can you improve them?

In [ ]:
github_activity = (
    metrics.filter(
        pl.col("project").is_in(["plotnine", "ggplot2", "shiny-python", "great-tables", "positron"]),
        pl.col("source") == "github",
        pl.col("metric").is_in(["daily_issues_opened", "daily_prs_merged", "daily_forks"]),
    )
    .group_by("project", "metric")
    .agg(pl.col("value").sum().alias("total"))
)

ggplot(github_activity, aes(x="project", fill="metric", y="total")) + geom_col()

**Optional bonus question:**

* Use the right function to manually specify your favorite hex colors or color names.

**Hints** *(highlight to reveal)*

<div style="color: #555; background-color: #555; padding: 8px; border-radius: 4px;">
&bull; Use a function that starts with <code>scale_fill_*</code>.<br>
&bull; Check the cheat sheet and Plotnine documentation for available functions.<br>
</div>

**Answer** *(highlight to reveal)*

<div style="color: #555; background-color: #555; padding: 8px; border-radius: 4px;">
<pre style="color: inherit; background: inherit; margin: 4px 0; padding: 0;">(
    ggplot(github_activity, aes(x="project", fill="metric", y="total"))
    + geom_col()
    + scale_fill_brewer(type="qual", palette="Set2")
)</pre>
Or with manual colors:<br>
<br>
<pre style="color: inherit; background: inherit; margin: 4px 0; padding: 0;">(
    ggplot(github_activity, aes(x="project", fill="metric", y="total"))
    + geom_col()
    + scale_fill_manual(values=("#e41a1c", "#377eb8", "#4daf4a"))
)</pre>
</div>

## Exercise 4: Layers and inheritance

The DataFrame `spikes` contains only the days where Plotnine had an unusually high number of downloads (more than 150,000).
Modify the line plot below so that these spike days appear as red points:

In [ ]:
plotnine_downloads = downloads.filter(pl.col("project") == "plotnine")
spikes = plotnine_downloads.filter(pl.col("value") > 150_000)
spikes

In [ ]:
ggplot(plotnine_downloads, aes(x="date", y="value")) + geom_line()

<div style="color: #555; background-color: #555; padding: 8px; border-radius: 4px;">
<b>Solution:</b><br>
1. Take the derivative: f'(x) = 2x<br>
2. Set f'(x) = 0, so x = 0
</div>
*(Drag cursor over box to reveal)*

## Exercise 5: Theme to your heart's content

Use one of the `theme_*()` functions and optionally the `theme()` function to change the looks of this plot:

In [ ]:
ggplot2_activity = metrics.filter(
    pl.col("project") == "ggplot2",
    pl.col("source") == "github",
    pl.col("metric").is_in(["daily_issues_opened", "daily_prs_merged", "daily_forks", "daily_comments"]),
)

(
    ggplot(ggplot2_activity, aes(x="date", y="value"))
    + geom_line()
    + facet_wrap("metric", scales="free_y")
)

**Optional bonus question:**

* Can you add a smoothed trend line using `geom_smooth()`?
* Try adjusting `panel_spacing`, `strip_text`, and `axis_text` inside `theme()`.

**Hints** *(highlight to reveal)*

<div style="color: #555; background-color: #555; padding: 8px; border-radius: 4px;">
&bull; Use a function like <code>theme_minimal()</code> or <code>theme_light()</code> as a starting point.<br>
&bull; Use <code>theme()</code> to override individual elements.<br>
&bull; Use <code>labs()</code> to add a title.<br>
</div>

**Answer** *(highlight to reveal)*

<div style="color: #555; background-color: #555; padding: 8px; border-radius: 4px;">
<pre style="color: inherit; background: inherit; margin: 4px 0; padding: 0;">(
    ggplot(ggplot2_activity, aes(x="date", y="value"))
    + geom_line(alpha=0.3)
    + geom_smooth(method="lowess", color="steelblue", se=False)
    + facet_wrap("metric", scales="free_y")
    + labs(title="ggplot2 GitHub Activity", x="", y="")
    + theme_minimal()
    + theme(
        strip_text=element_text(size=11, weight="bold"),
        panel_spacing=0.3,
        plot_title=element_text(size=14, margin={"b": 15}),
        dpi=144,
    )
)</pre>
</div>